# Notebook 03 — Lucas Critique Analysis

This is the **core experiment**: we investigate whether regime-switching models
are susceptible to the Lucas critique.

## The Lucas Critique in this context

Lucas (1976) argued that econometric models fail under policy changes because
the parameters they estimate are themselves functions of agents' expectations
about policy.  Here we operationalise this by:

1. **Training** all models on pre-break data from a Markov-switching DGP.
2. **Shifting** the DGP parameters (simulating a policy/structural change).
3. **Evaluating** models on post-break data from the new DGP.
4. **Measuring** the Lucas Sensitivity Ratio:

    $$\text{LSR} = \frac{\text{RMSE}_{\text{post}}}{\text{RMSE}_{\text{pre}}}$$

    LSR = 1 means no degradation.  LSR > 1 means Lucas-critique vulnerability.

**Key questions**
- Do classical models (MSM, TAR) have higher LSR than ML models?
- Does the severity of the shift affect the ranking of models?
- Do CUSUM and Chow tests correctly detect the break?

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from simulation import MarkovSwitchingDGP
from simulation.lucas_shift import MILD_SHIFT, SEVERE_SHIFT, simulate_pre_post_break, concatenate_periods
from models import HMMRegimeModel, ThresholdModel, MLRegimeModel, MixtureOfExpertsModel
from evaluation.metrics import (
    forecast_rmse, forecast_mae, directional_accuracy,
    regime_accuracy, adjusted_rand_regime, lucas_sensitivity_ratio,
)
from evaluation.lucas_critique import (
    chow_test, recursive_cusum,
    compute_rolling_performance, compare_pre_post_performance,
)
from evaluation.visualization import (
    plot_lucas_critique_results, plot_rolling_rmse,
    plot_regime_transition_heatmap, save_figure,
)

sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore')
print('Setup complete.')

## 1. Full Experiment via Pipeline

In [ ]:
from pipeline import LucasCritiqueExperiment

dgp = MarkovSwitchingDGP(seed=42)
experiment = LucasCritiqueExperiment(
    dgp=dgp,
    shift=MILD_SHIFT,
    n_pre=500,
    n_post=250,
    include_msm=True,
    run_chow=True,
)
results = experiment.run(verbose=True)

## 2. Summary Table

In [ ]:
summary = results.summary.copy()
print('\nFull results summary:')
print(summary.to_string(index=False, float_format='{:.4f}'.format))
print(f'\nBest model (lowest LSR): {results.best_model("LSR")}')
print(f'Best regime accuracy (pre-break): {results.best_model("pre_regime_acc")}')

## 3. Lucas Critique Visualisation

In [ ]:
from evaluation.lucas_critique import compare_pre_post_performance

comparison_df = summary[['model', 'pre_rmse', 'post_rmse', 'LSR']]
fig = plot_lucas_critique_results(comparison_df)
save_figure(fig, '03_lucas_critique_results')
plt.show()

## 4. Rolling RMSE: Performance Over Time

In [ ]:
# Re-fit models on pre-break, get predictions on full concatenated series
df_pre = results.df_pre
df_post = results.df_post
df_full = concatenate_periods(df_pre, df_post)
break_idx = len(df_pre)

# Fit all models
model_dict = {
    'HMM': HMMRegimeModel(n_components=2, random_state=42),
    'Threshold (TAR)': ThresholdModel(),
    'ML Regime (XGB)': MLRegimeModel(n_regimes=2),
    'Mixture of Experts': MixtureOfExpertsModel(n_experts=2, n_iter=50, random_state=42),
}
try:
    from models import MarkovSwitchingModel
    model_dict['Markov Switching (MSM)'] = MarkovSwitchingModel(k_regimes=2)
except Exception:
    pass

full_predictions = {}
for name, model in model_dict.items():
    try:
        model.fit(df_pre)
        # Predict on full series (models generalise out-of-sample via their learned params)
        y_pred_full = model.predict(df_full)
        full_predictions[name] = y_pred_full
        print(f'{name}: predicted {len(y_pred_full)} obs')
    except Exception as e:
        print(f'{name} failed: {e}')

In [ ]:
y_full = df_full['y'].to_numpy()
WINDOW = 40

rolling_dict = {}
for name, y_pred in full_predictions.items():
    rolling_dict[name] = compute_rolling_performance(y_full, y_pred, window=WINDOW)

ax = plot_rolling_rmse(rolling_dict, break_index=break_idx, window=WINDOW)
ax.set_title(f'Rolling RMSE (window={WINDOW}): Performance Degradation at Break Point')
save_figure(ax.get_figure(), '03_rolling_rmse')
plt.show()

## 5. Chow Test and CUSUM

In [ ]:
if results.chow is not None:
    print('Chow Structural Break Test:')
    for k, v in results.chow.items():
        print(f'  {k}: {v}')
    print()
    if results.chow['reject_H0']:
        print('Conclusion: The Chow test REJECTS parameter stability (structural break detected).')
    else:
        print('Conclusion: The Chow test FAILS TO REJECT parameter stability.')

In [ ]:
# CUSUM test on the concatenated AR(1) residuals
y_full_arr = df_full['y'].to_numpy()
X_full = df_full[['y_lag1']].assign(const=1.0)[['const', 'y_lag1']].to_numpy()

cusum_result = recursive_cusum(y_full_arr, X_full)

fig, ax = plt.subplots(figsize=(13, 4))
t_cusum = np.arange(len(cusum_result['cusum']))
ax.plot(t_cusum, cusum_result['cusum'], color='steelblue', lw=1.5, label='CUSUM')
ax.plot(t_cusum, cusum_result['upper_bound'], 'r--', lw=1, label='5% critical bound')
ax.plot(t_cusum, cusum_result['lower_bound'], 'r--', lw=1)
ax.fill_between(t_cusum, cusum_result['lower_bound'], cusum_result['upper_bound'],
                alpha=0.08, color='red')
ax.axvline(break_idx, color='black', ls=':', lw=1.5, label=f'Break point (t={break_idx})')
ax.axhline(0, color='gray', lw=0.7)
ax.set_xlabel('t')
ax.set_ylabel('CUSUM of recursive residuals')
ax.set_title('CUSUM Test for Parameter Stability')
ax.legend(fontsize=9)
save_figure(fig, '03_cusum_test')
plt.show()

print(f'Break detected: {cusum_result["break_detected"]}')
if cusum_result['break_index'] is not None:
    print(f'First exit from band at index: {cusum_result["break_index"]}')

## 6. Severe Shift Comparison

In [ ]:
print('Running experiment with SEVERE Lucas shift...')
exp_severe = LucasCritiqueExperiment(
    dgp=MarkovSwitchingDGP(seed=42),
    shift=SEVERE_SHIFT,
    n_pre=500,
    n_post=250,
    include_msm=True,
)
results_severe = exp_severe.run(verbose=False)

print('\nMild shift LSRs:')
print(summary[['model', 'LSR']].to_string(index=False, float_format='{:.3f}'.format))
print('\nSevere shift LSRs:')
print(results_severe.summary[['model', 'LSR']].to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# Merged comparison
merged = (
    summary[['model', 'LSR']].rename(columns={'LSR': 'LSR_mild'})
    .merge(results_severe.summary[['model', 'LSR']].rename(columns={'LSR': 'LSR_severe'}), on='model')
)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(merged))
width = 0.35
ax.bar(x - width/2, merged['LSR_mild'], width, label='Mild shift', color='#1f77b4')
ax.bar(x + width/2, merged['LSR_severe'], width, label='Severe shift', color='#d62728')
ax.axhline(1.0, color='black', ls='--', lw=1.2, label='No degradation (LSR=1)')
ax.set_xticks(x)
ax.set_xticklabels(merged['model'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Lucas Sensitivity Ratio')
ax.set_title('LSR: Mild vs Severe Lucas Shift')
ax.legend(fontsize=9)
plt.tight_layout()
save_figure(fig, '03_mild_vs_severe_lsr')
plt.show()

## 7. Interpretation

In [ ]:
best_mild = results.best_model('LSR')
best_severe = results_severe.best_model('LSR')
worst_mild = summary.sort_values('LSR', ascending=False).iloc[0]['model']

print('Key findings:')
print(f'  Under mild shift  — most Lucas-stable model: {best_mild}')
print(f'  Under severe shift — most Lucas-stable model: {best_severe}')
print(f'  Worst (most Lucas-vulnerable, mild shift): {worst_mild}')
print()
print('Discussion:')
print('''  The Lucas Sensitivity Ratio measures how much a model's forecast
  performance degrades once the DGP parameters shift.  Classical models
  (Markov Switching, TAR) fix their parameters at estimation time, making
  them inherently brittle to structural change.  ML models (XGBoost regime
  switcher, Mixture of Experts) may show lower LSR if their learned
  representations are robust to moderate parameter drift — but they are
  not immune.  Under a severe shift, the clusters and gating functions
  fitted to the pre-break regime structure become equally outdated.

  This experiment illustrates that the Lucas Critique is a fundamental
  epistemic problem, not purely a modelling artefact: any model that learns
  from historical data is susceptible when the structural environment changes.''')

## 8. Save Final Results

In [ ]:
data_dir = PROJECT_ROOT / 'data' / 'simulated'
results.summary.to_csv(data_dir / 'lucas_critique_mild.csv', index=False)
results_severe.summary.to_csv(data_dir / 'lucas_critique_severe.csv', index=False)
merged.to_csv(data_dir / 'lucas_critique_comparison.csv', index=False)
print('Results saved.')